# Notebook 16 — NCCS Baseline Evaluation, Leakage Audit, Freeze and Handoff

**Purpose:** Close out the exploratory Form 990 NCCS baseline prior to Databricks and STTR. This notebook **does not train or retune models**. It audits source artifacts, compares the supervised ranking with size-only baselines and within-size groups, documents supervised/peer differences, produces a fixed baseline export, and generates a machine-readable audit/manifest.

**Interpretation:** `fraud_label` is a **self-reported significant diversion disclosure**, not confirmed fraud. Scores are not calibrated probabilities of fraud. The 2023–2024 period was previously examined and must **not** be called a pristine untouched holdout. Size associations are not proof of bias. Case profiles are research prompts, not allegations. The deliberately balanced 100-case review sample is not population representative.

**Security:** Organization-identifiable profiles and score exports are **restricted analyst/demo inputs**. Do not publish case-level narratives or distribute reviewer identity mappings. Do not mix label/outcome columns into prospective scoring feature tables.

**Execution:** Run from repository root or `notebooks/` with the `irs990` environment, after Notebooks 09–15. Outputs go only to `data/processed/notebook16/`. No prior artifacts are overwritten. Review all warnings and unresolved audit findings before approving the baseline.

**Execution status:** Notebook 16 has been executed successfully against the local NCCS dataset. The 2024 evaluation labels were recovered and validated, the size-only and within-size comparisons were completed, and the baseline scoring artifacts and case profiles were snapshotted. Subsequent analyst review documented model differences, feature-lineage findings, evaluation limitations, and the recommended exploratory baseline decision.

## 1. Configuration, provenance and prerequisites

In [ ]:
from pathlib import Path
import json, hashlib, re, shutil, datetime, sys
import numpy as np
import pandas as pd
from IPython.display import display

ROOT=Path.cwd().resolve()
if ROOT.name=='notebooks': ROOT=ROOT.parent
DATA=ROOT/'data'/'processed'
OUT=DATA/'notebook16'
RESTRICTED=OUT/'restricted_handoff'
OUT.mkdir(parents=True,exist_ok=True)
RESTRICTED.mkdir(parents=True,exist_ok=True)
PRIMARY=DATA/'notebook12/07_2024_organization_primary_queue.parquet'
LIMITED=DATA/'notebook12/07_2024_organization_limited_data_queue.parquet'
SIZE=DATA/'notebook14/09_2024_size_robustness_research_queue.parquet'
PROFILES=DATA/'notebook15/restricted_analyst_only/03_RESTRICTED_sample_case_profiles.parquet'
BLINDED=DATA/'notebook15/blinded_reviewer_packet/02_blinded_case_profiles.csv'
MODEL=DATA/'notebook09/selected_model.joblib'
MODEL_MANIFEST=DATA/'notebook09/model_manifest.json'
FEATURES=DATA/'nccs_longitudinal_features_v1.parquet'
REQUIRED=[PRIMARY,LIMITED,SIZE,PROFILES,MODEL,MODEL_MANIFEST]
missing=[str(p) for p in REQUIRED if not p.is_file()]
assert not missing, 'Run prerequisite notebooks / verify paths: '+str(missing)

def sha256(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for chunk in iter(lambda:f.read(4*1024*1024),b''):h.update(chunk)
    return h.hexdigest()
def save_json(name,obj):
    (OUT/name).write_text(json.dumps(obj,indent=2,default=str))
print('Root:',ROOT)
print('Inputs present:',len(REQUIRED))

## 2. Load frozen organization queues and verify grain

In [ ]:
p=pd.read_parquet(PRIMARY)
l=pd.read_parquet(LIMITED)
s=pd.read_parquet(SIZE)
assert 'ein' in p and 'ein' in l and 'ein' in s
assert p.ein.notna().all() and p.ein.is_unique
assert l.ein.is_unique and not set(p.ein).intersection(l.ein)
assert s.ein.is_unique and set(s.ein)==set(p.ein), 'Notebook 14 population differs from Notebook 12'
assert len(p)>0
needed=['organization_supervised_rank','organization_peer_rank','organization_consensus_rank','supervised_diversion_score','peer_adverse_6','consensus_score','revenue']
assert not set(needed)-set(p.columns),f'Missing columns: {set(needed)-set(p.columns)}'
print('Primary:',len(p),'Limited:',len(l),'Size diagnostic:',len(s))
print('Columns:',sorted(p.columns.tolist()))
print('2024 primary unique EINs:',p.ein.nunique())

## 3. Recover observed 2024 disclosure labels safely

The Notebook 12 organization queue omits `fraud_label`. Recover it from the canonical NCCS longitudinal dataset. Join on **OBJECTID + EIN + tax_year** when possible, with EIN normalization; do not silently choose among conflicting filings. If `OBJECTID` cannot match, use EIN + tax year **only when that source key is unique**. Never turn missing disclosures into negatives.

The disclosure label is for **retrospective evaluation only**, not model scoring, training, or the Databricks scoring export.

In [ ]:
# Read only the necessary columns and only 2024 to limit memory use.
assert FEATURES.is_file(), f"Missing canonical longitudinal dataset: {FEATURES}"
import duckdb
con=duckdb.connect()
con.execute("SET memory_limit='6GB'")
con.execute("SET threads=4")
escaped=str(FEATURES).replace("'","''")
source_columns=set(con.execute(f"DESCRIBE SELECT * FROM read_parquet('{escaped}')").df().column_name)
required_source={'ein','tax_year','fraud_label'}
assert required_source <= source_columns, f"Canonical source missing {required_source-source_columns}"
source_fields=['ein','tax_year','fraud_label']+(['OBJECTID'] if 'OBJECTID' in source_columns else [])
source_sql=', '.join('"'+c+'"' for c in source_fields)
labels=con.execute(f"SELECT {source_sql} FROM read_parquet('{escaped}') WHERE TRY_CAST(tax_year AS INTEGER)=2024").df()
con.close()

def canonical_ein(v):
    return (v.astype('string').str.strip().str.upper()
            .str.replace(r'^EIN-', '', regex=True)
            .str.replace(r'[^0-9]', '', regex=True).str.zfill(9))
def canonical_objectid(v):
    return v.astype('string').str.strip().str.replace(r'\.0$', '', regex=True)

labels['_ein_key']=canonical_ein(labels['ein'])
labels['_year_key']=pd.to_numeric(labels['tax_year'],errors='coerce').astype('Int64')
labels['_label']=pd.to_numeric(labels['fraud_label'],errors='coerce')
assert labels['_label'].dropna().isin([0,1]).all(), 'Unexpected disclosure label values'
assert labels['_ein_key'].str.fullmatch(r'\d{9}').fillna(False).all(), 'Invalid source EINs'
q=p.copy()
q['_ein_key']=canonical_ein(q['ein'])
q['_year_key']=pd.to_numeric(q['tax_year'],errors='coerce').astype('Int64') if 'tax_year' in q else pd.Series(2024,index=q.index,dtype='Int64')
assert q['_year_key'].eq(2024).all(), 'Primary queue contains non-2024 rows'
assert q['_ein_key'].str.fullmatch(r'\d{9}').fillna(False).all()
assert q['_ein_key'].is_unique, 'Primary EIN key collision after normalization'

join_cols=['_ein_key','_year_key']
if 'OBJECTID' in labels and 'OBJECTID' in q:
    labels['_object_key']=canonical_objectid(labels['OBJECTID'])
    q['_object_key']=canonical_objectid(q['OBJECTID'])
    # Exact filing join only if every queue OBJECTID is populated and the source key is unique.
    if (q['_object_key'].notna().all() and q['_object_key'].ne('').all()
            and not labels.duplicated(join_cols+['_object_key']).any()):
        join_cols+=['_object_key']

if labels.duplicated(join_cols).any():
    conflicts=labels.loc[labels.duplicated(join_cols,keep=False),join_cols+['_label']]
    conflicts.to_csv(RESTRICTED/'01_RESTRICTED_ambiguous_source_labels.csv',index=False)
    raise ValueError(f"Ambiguous canonical labels on {join_cols}; review restricted diagnostic; do not silently deduplicate.")

eval_df=q.merge(labels[join_cols+['_label']],on=join_cols,how='left',validate='one_to_one',indicator='_label_join')
assert len(eval_df)==len(p) and eval_df.ein.is_unique
join_summary={'join_keys':join_cols,'source_2024_rows':len(labels),
              'primary_rows':len(p),'matched_rows':int(eval_df._label_join.eq('both').sum()),
              'known_labels':int(eval_df._label.isin([0,1]).sum()),
              'positive_disclosures':int(eval_df._label.eq(1).sum()),
              'unknown_labels':int(eval_df._label.isna().sum())}
save_json('01_label_join_diagnostics.json',join_summary)
display(pd.DataFrame([join_summary]))
assert join_summary['matched_rows']==len(p), 'Some primary EINs did not match the canonical source; inspect join keys'
assert join_summary['known_labels']>0 and join_summary['positive_disclosures']>0, 'No evaluable positives'
eval_known=eval_df.loc[eval_df._label.isin([0,1])].copy()
# Outcome data are stored separately in restricted storage only.
eval_df[['ein','_label']].rename(columns={'_label':'fraud_label'}).to_parquet(
    RESTRICTED/'01_RESTRICTED_2024_evaluation_labels.parquet',index=False)
print('Known-label cohort:',len(eval_known),'positives:',int(eval_known._label.sum()),
      'prevalence:',round(eval_known._label.mean(),6))

def rank_metrics(df,rank_col,ks=(100,500,1000)):
    valid=df.dropna(subset=[rank_col,'_label']).sort_values(rank_col,kind='stable')
    n=len(valid); positives=int(valid._label.sum()); base=positives/n if n else np.nan
    rows=[]
    for k in ks:
        sample=valid.head(min(k,n)); found=int(sample._label.sum())
        rows.append(dict(rank=rank_col,k=k,reviewed=len(sample),found=found,
            precision=found/len(sample) if len(sample) else np.nan,
            recall=found/positives if positives else np.nan,
            prevalence=base,lift=(found/len(sample))/base if base and len(sample) else np.nan))
    return rows
rows=[]
for col in ['organization_supervised_rank','organization_peer_rank','organization_consensus_rank']:
    rows+=rank_metrics(eval_known,col)
metrics=pd.DataFrame(rows)
display(metrics)
metrics.to_csv(OUT/'01_2024_retrospective_ranking_metrics.csv',index=False)

## 4. Revenue-only and assets-only benchmarks

Compare the original supervised ranking with a simple descending-size ranking **on the same known-label, size-observed cohort**. Missing size values are excluded symmetrically. These comparisons are descriptive and retrospective, not fresh held-out validation.

In [ ]:
size_rows=[]
for field in ['revenue','assets']:
    if field not in eval_known.columns:
        print('Unavailable size field:',field)
        continue
    tmp=eval_known.copy()
    tmp['_size']=pd.to_numeric(tmp[field],errors='coerce')
    tmp=tmp.loc[tmp['_size'].notna()].copy()
    assert len(tmp)>0
    tmp['size_only_rank']=tmp['_size'].rank(ascending=False,method='first')
    tmp['supervised_subset_rank']=pd.to_numeric(tmp['organization_supervised_rank'],errors='coerce').rank(method='first')
    for name,col in [('size_only', 'size_only_rank'),('supervised_same_eligible_cohort','supervised_subset_rank')]:
        for row in rank_metrics(tmp,col):
            row.update(comparator=name,size_field=field,eligible_n=len(tmp),
                       eligible_positives=int(tmp._label.sum()))
            size_rows.append(row)
size_compare=pd.DataFrame(size_rows)
assert not size_compare.empty, 'No size comparisons generated'
display(size_compare)
size_compare.to_csv(OUT/'02_size_only_vs_supervised.csv',index=False)


## 5. Within-size-group evaluation
Use the size strata **already constructed in Notebook 14**, rather than silently changing the boundaries. Report within-group prevalence and supervised top-K enrichment. Small positive counts can make estimates highly unstable.

In [ ]:
assert 'size_stratum' in s.columns
g=eval_known[['ein','_label','organization_supervised_rank','organization_peer_rank']].merge(
    s[['ein','size_stratum']],on='ein',validate='one_to_one')
assert len(g)==len(eval_known)
within=[]
for group,df in g.groupby('size_stratum',dropna=False,observed=True):
    for method,col in [('supervised','organization_supervised_rank'),('peer','organization_peer_rank')]:
        local=df.copy()
        local['_within_rank']=pd.to_numeric(local[col],errors='coerce').rank(method='first')
        for k in (25,100,500):
            for row in rank_metrics(local,'_within_rank',ks=(k,)):
                row.update(size_stratum=str(group),method=method,group_n=len(local),
                           group_positives=int(local._label.sum()))
                within.append(row)
within_df=pd.DataFrame(within)
assert not within_df.empty
display(within_df)
within_df.to_csv(OUT/'03_within_size_group_performance.csv',index=False)
print('Note: small positive counts make within-stratum precision and lift unstable.')


## 6. Supervised vs peer ranking differences and representative case selection
The case sample is **stratified for comparison**, not for population estimates. We select examples reproducibly from Notebook 15, without claiming they are confirmed misconduct. Restricted examples should be reviewed by an analyst before use in the demo narrative.

In [ ]:
top_k=1000
supervised=set(p.nsmallest(top_k,'organization_supervised_rank').ein)
peer=set(p.nsmallest(top_k,'organization_peer_rank').ein)
overlap={'top_k':top_k,'supervised_only':len(supervised-peer),'peer_only':len(peer-supervised),'both':len(supervised&peer),'jaccard':len(supervised&peer)/len(supervised|peer)}
print(overlap)
profiles=pd.read_parquet(PROFILES)
print('Profile columns:',sorted(profiles.columns.tolist()))
assert 'ein' in profiles and profiles.ein.is_unique
# The Notebook 13/15 review sample intentionally includes four groups.
if 'validation_signal_pattern' in profiles:
    display(profiles.validation_signal_pattern.value_counts(dropna=False))
    examples=profiles.sort_values('ein').groupby('validation_signal_pattern',dropna=False,group_keys=False).head(2)
else:
    print('WARNING: signal-group label missing; selecting deterministic examples without group stratification')
    examples=profiles.sort_values('ein').head(8)
examples.to_csv(RESTRICTED/'04_RESTRICTED_representative_case_examples.csv',index=False)
summary={'top1000_overlap':overlap,'sample_profiles':len(profiles),'representative_examples':len(examples),'interpretation':['Supervised ranks association with self-reported diversion disclosures.','Peer ranks unusual financial/governance indicators relative to peers.','Low overlap may mean complementary signals or unhelpful ranking differences; human review is pending.','Selected profiles are examples, not proof of misconduct.']}
save_json('04_model_difference_summary.json',summary)
print('Wrote restricted representative cases:',len(examples))

# Produce a factual, non-accusatory profile comparison, not a claim of lead quality.
profile_numeric=[c for c in ['revenue','assets','expenses','liabilities',
                              'six_indicator_count','supervised_diversion_score',
                              'peer_adverse_6','consensus_score'] if c in profiles.columns]
group_col='validation_signal_pattern' if 'validation_signal_pattern' in profiles else None
if group_col:
    group_summary=profiles.groupby(group_col,dropna=False)[profile_numeric].median(numeric_only=True)
    display(group_summary)
    group_summary.to_csv(RESTRICTED/'04_RESTRICTED_profile_group_medians.csv')
case_lines=['# Representative case comparisons (analyst-only)',
            'These cases were deliberately sampled by model-selection group; they are not representative of the population.',
            'The numbers and pre-existing observed patterns below are descriptive, not evidence of misconduct.',
            'No investigator has rated lead quality.']
for _,case in examples.iterrows():
    case_lines.append('\n## Review ID: '+str(case.get('review_id','not available'))+
                      ' | Selection group: '+str(case.get('validation_signal_pattern','unavailable')))
    for field in profile_numeric:
        value=case.get(field)
        if pd.notna(value):
            case_lines.append(f'- {field}: {value}')
    for field in ['observed_patterns','top_peer_explanations','possible_nonfraud_explanations']:
        if field in case and pd.notna(case[field]):
            case_lines.append(f'- {field}: {str(case[field])[:1500]}')
    case_lines.append('- Analyst interpretation: pending review of the underlying filing and context.')
(RESTRICTED/'04_RESTRICTED_representative_case_comparison.md').write_text(
    '\n'.join(case_lines),encoding='utf-8')
print('Wrote factual representative case comparison. Human interpretation remains pending.')


## 7. Evaluation design and outcome-leakage audit
This audit checks the saved feature manifest and performs conservative name-based screening. **A clean name scan does not prove absence of leakage**: review upstream feature engineering, timestamp availability, joins, label derivation, and whether a field indirectly encodes the outcome. Any unresolved issue must remain visible in the manifest.

In [ ]:
manifest=json.loads(MODEL_MANIFEST.read_text())
features=manifest.get('features',[])
assert isinstance(features,list) and features, 'No explicit feature list in model manifest; cannot approve freeze'
# Explicit label/answer field checks, with suspicious feature names routed to manual review.
forbidden={'fraud_label','significant_diversion','significant_diversion_of_assets','diversion_disclosure','diversion_flag','target','label','outcome'}
exact_hits=[x for x in features if x.lower() in forbidden]
pattern=re.compile(r'diversion|fraud|investigat|enforc|criminal|disclos|outcome|label|target|case_status|audit_result',re.I)
review_hits=[x for x in features if pattern.search(x)]
leakage={'feature_count':len(features),'exact_outcome_feature_hits':exact_hits,'name_based_manual_review_hits':review_hits,'feature_names':features,'model_manifest_path':str(MODEL_MANIFEST.relative_to(ROOT)),'audit_scope':'Manifest-level static screening only; lineage and point-in-time availability require independent verification','status':'FAIL' if exact_hits else 'REQUIRES_MANUAL_LINEAGE_AND_TIMING_REVIEW'}
save_json('05_leakage_feature_audit.json',leakage)
print('Exact forbidden:',exact_hits,'Manual review candidates:',review_hits)
assert not exact_hits,'Outcome-revealing predictor found. Do not freeze as approved.'
# Explicitly document known limitations rather than asserting a pristine holdout.
evaluation={'training_years':manifest.get('train_years','NOT RECORDED'), 'validation_years':manifest.get('validation_years','NOT RECORDED'),'later_evaluation_years':manifest.get('test_years','NOT RECORDED; 2023–2024 previously examined'),'untouched_holdout_confirmed':False,'feature_leakage_proven_absent':False,'point_in_time_availability_confirmed':False,'known_issue':'Outcome is self-reported diversion, not confirmed fraud. Training/scoring population and publication lags require review.','action':'Verify Notebook 09 split implementation, source feature construction, and point-in-time availability before approval.'}
save_json('05_evaluation_design_and_caveats.json',evaluation)
display(pd.DataFrame({'finding':['Exact outcome-name predictors','Potentially sensitive predictor names','Pristine untouched holdout','Feature lineage and point-in-time verification'],'result':[str(exact_hits),str(review_hits),'NOT ESTABLISHED','MANUAL REVIEW REQUIRED']}))

**Manual audit update:** Following execution, the feature-engineering and modeling code in Notebooks 08 and 09 was reviewed. No obvious direct outcome leakage or future-tax-year feature leakage was identified. The automated audit output above represents the original manifest-level screening and does not incorporate this subsequent manual review. Publication-date availability, prospective validation, and formal baseline approval remain outstanding.

## 8. Create Protected Baseline Snapshot and Restricted Databricks/STTR Handoff
This is an **artifact snapshot**, not a claim of model validation. Freeze copies of the selected model and its manifest, keep component scores and rankings separate, and export a prospective scoring table **without the outcome label**. Outcomes are retained only in a separately labeled restricted evaluation file if available.

In [ ]:

# Section 8 — Baseline Freeze
# Protect baseline_v1 from accidental overwrites and keep
# observed outcomes separate from general scoring data.

import os
import tempfile
import shutil
from pathlib import Path


# ------------------------------------------------------------
# 1. Establish baseline snapshot directory
# ------------------------------------------------------------

SNAP = RESTRICTED / "baseline_v1"
SNAP.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 2. Helper: safely write Parquet files
# ------------------------------------------------------------

def write_parquet_protected(df, destination):
    """
    Create a Parquet file without overwriting an existing
    baseline artifact.

    If the file exists:
      - Identical bytes: retain the original.
      - Different bytes: raise an error.

    If the file does not exist:
      - Create it exclusively.
    """

    destination = Path(destination)

    with tempfile.NamedTemporaryFile(
        suffix=".parquet",
        dir=destination.parent,
        delete=False
    ) as temp:
        temp_path = Path(temp.name)

    try:
        df.to_parquet(temp_path, index=False)

        if destination.exists():
            if sha256(destination) != sha256(temp_path):
                raise RuntimeError(
                    f"Protected baseline file differs: {destination}. "
                    "Do not overwrite baseline_v1. "
                    "Investigate the difference or create baseline_v2."
                )

            print(f"Verified existing file: {destination.name}")
            return

        # Exclusive creation prevents replacing an existing file.
        with open(temp_path, "rb") as source:
            with open(destination, "xb") as target:
                shutil.copyfileobj(source, target)

        print(f"Created protected file: {destination.name}")

    finally:
        temp_path.unlink(missing_ok=True)


# ------------------------------------------------------------
# 3. Protect model and model manifest
# ------------------------------------------------------------

for src in [MODEL, MODEL_MANIFEST]:
    dst = SNAP / src.name

    if dst.exists():
        assert sha256(dst) == sha256(src), (
            f"Existing frozen artifact changed: {dst}"
        )
    else:
        shutil.copy2(src, dst)


# ------------------------------------------------------------
# 4. Construct general scoring export
# ------------------------------------------------------------

# Restrict exported columns to stable identifiers,
# model components, indicators, and financial context.

base_cols = [
    "ein",
    "OBJECTID",
    "tax_year",
    "peer_group",
    "revenue",
    "assets",
    "expenses",
    "liabilities",
    "six_indicator_count",
    "supervised_diversion_score",
    "peer_adverse_6",
    "consensus_score",
    "organization_supervised_rank",
    "organization_peer_rank",
    "organization_consensus_rank",
    "organization_isolation_rank",
    "top_peer_explanations"
]

score_cols = [c for c in base_cols if c in p.columns]

assert all(
    x in score_cols
    for x in [
        "ein",
        "supervised_diversion_score",
        "peer_adverse_6",
        "consensus_score"
    ]
)

score = p[score_cols].copy()


# Optionally include available peer indicator fields,
# but never outcome or target fields.

peer_candidates = [
    c for c in p.columns
    if any(
        t in c.lower()
        for t in [
            "expense_to_revenue",
            "liabilit",
            "revenue_growth",
            "asset_growth",
            "revenue_volatility",
            "board_indep"
        ]
    )
]

peer_candidates = [
    c for c in peer_candidates
    if c not in score.columns
    and not re.search(
        r"diversion|fraud_label|outcome|target|disclos",
        c,
        re.I
    )
]

for col in peer_candidates:
    score[col] = p[col]


# ------------------------------------------------------------
# 5. VERIFY SCORING / OUTCOME SEPARATION
#    This is the new check you asked about.
# ------------------------------------------------------------

forbidden_fields = {
    "fraud_label",
    "_label",
    "label_status",
    "significant_diversion",
    "diversion_disclosure"
}

assert not forbidden_fields.intersection(score.columns), (
    "Outcome-revealing fields found in scoring export."
)

# Keep the existing broader pattern-based check.
assert not any(
    re.search(
        r"diversion(?!_score)|fraud_label|outcome|target|disclos",
        c,
        re.I
    )
    for c in score.columns
), "Potential outcome field found in scoring export."

# Verify one primary scoring row per organization.
assert score["ein"].notna().all(), (
    "Missing EIN in primary scoring export."
)

assert score["ein"].is_unique, (
    "Duplicate EIN values in primary scoring export."
)

# Verify one evaluation label per organization.
assert eval_df["ein"].notna().all(), (
    "Missing EIN in evaluation labels."
)

assert eval_df["ein"].is_unique, (
    "Duplicate EIN values in evaluation labels."
)

print("Scoring/outcome separation checks passed.")


# ------------------------------------------------------------
# 6. Export primary scores WITHOUT observed outcomes
# ------------------------------------------------------------

write_parquet_protected(
    score,
    SNAP / "2024_primary_baseline_scores_NO_OUTCOMES.parquet"
)


# ------------------------------------------------------------
# 7. Export evaluation outcomes SEPARATELY
# ------------------------------------------------------------

evaluation_outcomes = (
    eval_df[["ein", "_label"]]
    .rename(columns={"_label": "fraud_label"})
    .copy()
)

write_parquet_protected(
    evaluation_outcomes,
    SNAP / (
        "RESTRICTED_2024_evaluation_outcomes_"
        "DO_NOT_JOIN_TO_SCORING.parquet"
    )
)


# ------------------------------------------------------------
# 8. Preserve restricted case profiles
# ------------------------------------------------------------

# Case profiles may contain blinded lookup IDs,
# internal selection groups, and analyst narratives.

case_dst = SNAP / "RESTRICTED_sample_case_profiles.parquet"

if case_dst.exists():
    assert sha256(case_dst) == sha256(PROFILES), (
        f"Existing frozen case profiles changed: {case_dst}"
    )
    print("Verified existing case profiles.")
else:
    with open(PROFILES, "rb") as source:
        with open(case_dst, "xb") as target:
            shutil.copyfileobj(source, target)

    print("Created restricted case profiles.")


# ------------------------------------------------------------
# 9. Export limited-data population separately
# ------------------------------------------------------------

limited_cols = [
    c for c in [
        "ein",
        "OBJECTID",
        "tax_year",
        "six_indicator_count",
        "revenue",
        "assets"
    ]
    if c in l.columns
]

write_parquet_protected(
    l[limited_cols],
    SNAP / "2024_limited_data_coverage.parquet"
)


# ------------------------------------------------------------
# 10. Confirm baseline snapshot
# ------------------------------------------------------------

print()
print("Baseline snapshot:", SNAP)
print("Primary scores:", len(score))
print("Scoring columns:", len(score.columns))
print("Limited-data organizations:", len(l))
print("Restricted case profiles:", len(profiles))
print("Baseline overwrite protection enabled.")

## 9. Version manifest, QA gates, and handoff instructions
**Freeze status** must be interpreted as *technically snapshotted, pending explicit audit approval* until the leakage and evaluation review is signed off. The notebook cannot certify facts not observable from the artifacts.

In [ ]:
inputs=[PRIMARY,LIMITED,SIZE,PROFILES,MODEL,MODEL_MANIFEST,FEATURES]
outputs=sorted([f for f in OUT.rglob('*') if f.is_file() and f.name!='09_baseline_freeze_manifest.json'])
assert len(score)==len(p) and score.ein.is_unique
assert len(profiles)==profiles.ein.nunique()
assert not any(c in score.columns for c in ['fraud_label','diversion_disclosure'])
report={'baseline_id':'nccs_2024_baseline_v1','created_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
'manual_feature_lineage_review': (
    'Completed code-level review of Notebooks 08 and 09; '
    'no obvious direct outcome or future-tax-year leakage found. '
    'Publication-date validation and formal approval remain outstanding.'
),'purpose':'Frozen exploratory comparison baseline for future STTR experiments and Databricks demo','source_inputs':[{'path':str(f.relative_to(ROOT)),'sha256':sha256(f)} for f in inputs],'snapshot_and_reports':[{'path':str(f.relative_to(ROOT)),'sha256':sha256(f)} for f in outputs],'counts':{'primary':len(p),'limited':len(l),'case_profiles':len(profiles)},'model_components':['supervised_diversion_score','peer_adverse_6','consensus_score'],'consensus_weights':{'supervised_percentile':0.5,'peer_percentile':0.5},'weights_validated':False,'outcome':'self-reported significant diversion disclosure; not confirmed fraud','evaluation_holdout_status':'Previously examined 2023–2024; NOT pristine','leakage_audit':leakage['status'],'release_status':'SNAPSHOT_CREATED_EVALUATION_COMPUTED_PENDING_MANUAL_FEATURE_LINEAGE_APPROVAL','limitations':['No independently verified fraud outcomes','No human lead-quality reviews completed','100-case sample is deliberately balanced and not population representative','Organization size influences supervised rankings','2023–2024 already examined; no untouched prospective test','Feature-name screening cannot establish absence of indirect leakage or publication-timing issues'],'databricks_instructions':['Keep restricted case profiles and evaluation outcomes in controlled-access storage','Create a scoring Delta table from 2024_primary_baseline_scores_NO_OUTCOMES.parquet','Keep evaluation outcomes in separate restricted Delta table; never use them as scoring predictors','Load 2024_limited_data_coverage.parquet separately and label incomplete coverage','Validate row counts, EIN uniqueness, and schema after ingestion','Keep version/hash metadata for STTR comparisons','Do not present ranks as fraud probabilities or confirmed allegations']}
manifest_path = OUT / "09_baseline_freeze_manifest.json"

if manifest_path.exists():
    print("Existing baseline freeze manifest preserved:", manifest_path)
else:
    save_json("09_baseline_freeze_manifest.json", report)
    print("Created baseline freeze manifest:", manifest_path)
print('Technical snapshot created. Status:',report['release_status'])
print('Files:',len(outputs)+1)
print('Next: obtain formal baseline approval, then upload restricted exports to Databricks.')

### Within-Size Model Performance — Interpretation

To assess whether supervised model performance was driven primarily by organization size, we compared supervised and peer-based anomaly rankings within 10 revenue deciles and a separate nonpositive-revenue group.

Across the 11 size groups, the supervised model identified more self-reported significant asset-diversion disclosures than the peer-based model at all three evaluated review capacities. At the top 500 organizations per group, the supervised model identified 21 positive disclosures compared with 13 for the peer-based model. However, supervised performance was not consistently better across individual groups: it identified more disclosures in five groups, fewer in two, and the same number in four.

These findings suggest that the supervised model retains some disclosure-ranking signal within organization-size groups, rather than relying exclusively on differences in organization size. However, the number of positive disclosures is very small, making group-level performance estimates unstable. Furthermore, the supervised model did not outperform a simple revenue-only ranking in the overall top-K evaluation.

**Conclusion:** The supervised model demonstrates some within-size discrimination, but its incremental value over simple size-based prioritization remains unproven. Both supervised and peer-based anomaly signals should be retained as complementary exploratory baselines, pending further validation of their investigative usefulness.

### Representative Case-Profile Comparison — Interpretation

Eight representative organization profiles were reviewed across four model-selection groups: supervised-only, peer-anomaly-only, both models, and neither model. These cases were deliberately selected to illustrate differences in model behavior and are not representative of the overall nonprofit population.

The supervised-only examples included very large organizations. For instance, R0024 reported approximately $1.56 billion in revenue and $4.94 billion in assets but received a relatively low peer anomaly score (0.184). This is consistent with the broader observation that the supervised ranking is associated with organization size, although individual feature contributions have not been established.

The peer-only examples demonstrated more pronounced financial changes relative to comparable organizations. R0007 reported expenses exceeding revenue, revenue and asset declines of at least 50% between 2022 and 2024, and unusually high revenue volatility. Its peer anomaly score was 0.874 despite not appearing in the supervised top-priority group.

Organizations prioritized by both approaches exhibited multiple adverse financial indicators. R0049, for example, reported expenses substantially exceeding revenue, liabilities exceeding assets, and significant historical declines in revenue and assets. Conversely, R0051 was prioritized by neither model despite some unusual historical indicators, illustrating that individual financial signals do not necessarily translate into high overall model priority.

**Conclusion:** The supervised and peer-based models identify meaningfully different financial profiles. The supervised approach tends to prioritize organizations with characteristics associated with historically disclosed diversions, including large organizations, while the peer-based approach emphasizes unusual financial patterns relative to comparable nonprofits. Their disagreement may provide complementary perspectives for exploratory screening.

These findings describe model behavior, not investigative effectiveness. Unusual financial conditions may reflect legitimate operational circumstances, including grant timing, restricted funding, restructuring, accounting differences, or data-quality issues. No independent investigator has evaluated these cases, and the profiles do not establish fraud or misconduct.

### Feature-Lineage and Evaluation-Split Audit — Findings

The selected supervised baseline uses 18 financial, governance, historical-trend, and derived-indicator predictors. Review of the feature-engineering code in Notebook 08 confirmed that the asset-diversion disclosure field is used to construct the target (`fraud_label`) but is not directly included among the selected predictors. Financial and governance predictors originate from Form 990 summary and governance fields, while historical features use current-year and preceding-year observations without forward-looking tax-year windows.

Review of Notebook 09 confirmed the documented temporal evaluation design: training on tax years 2010–2020, validation on 2021–2022, and later evaluation on 2023–2024. Unknown disclosure labels are excluded from supervised fitting and evaluation, and model preprocessing is fitted using training data. Model selection is based on validation average precision.

**No direct outcome leakage or future-tax-year feature leakage was identified in the reviewed code.** However, the evaluation is retrospective and uses tax years rather than actual filing publication dates. Same-year financial and governance information is appropriate for screening already-filed returns but does not establish prospective prediction capability.

Additional limitations include duplicate-filing and label-resolution issues identified in earlier data preparation, repeated organizations across temporal splits, negative downsampling during training, and uncalibrated model scores. The 2023–2024 evaluation period has also been examined during subsequent development and is no longer a pristine independent holdout for future model comparisons.

**Audit conclusion:** Code-level review of the selected predictors, feature construction, and temporal split found no obvious direct outcome leakage or forward-looking tax-year calculations. The baseline is suitable for documented retrospective exploratory comparison. Full point-in-time validation, independent prospective evaluation, and resolution of outstanding data-quality limitations remain future work.

## 10. Summary and Baseline Decision

**Current state:** A reproducible exploratory NCCS baseline has been technically snapshotted, including separate supervised, peer-based anomaly, and 50/50 consensus scores. The supervised model targets self-reported significant asset-diversion disclosures, not verified fraud. Its operational investigative usefulness remains unvalidated.

**Observed-label evaluation:** All 329,440 organizations in the 2024 primary scoring population were matched to evaluation labels, including 296 positive diversion disclosures (approximately 0.09% prevalence). The supervised model identified 8 positive disclosures in its top 1,000 organizations, compared with 4 for the peer-based model and 7 for the consensus ranking.

**Size-only comparison:** A revenue-only ranking identified 9 positive disclosures in the top 1,000, compared with 8 for the supervised model. An assets-only ranking also identified 8. The supervised model therefore has not demonstrated incremental disclosure-prediction value over simple organization-size prioritization at the evaluated review capacities.

**Within-size evaluation:** Across 10 revenue deciles and a separate nonpositive-revenue group, the supervised model identified more positive disclosures than the peer-based model at all three evaluated review capacities. At the top 500 organizations per group, the supervised model identified 21 disclosures compared with 13 for the peer-based model. However, it performed better in only five groups, worse in two, and tied in four. Small positive counts limit the reliability of group-level performance comparisons.

**Model differences:** The supervised and peer-based models identified substantially different organizations, with only 32 organizations appearing in both top-1,000 rankings. Review of eight deliberately selected case profiles showed that supervised-only cases can include very large organizations with relatively low peer anomaly scores, while peer-only cases often exhibit pronounced financial changes relative to comparable nonprofits. Cases prioritized by both methods frequently displayed multiple adverse financial indicators. These findings describe complementary screening perspectives, not verified investigative usefulness.

**Feature-lineage and evaluation audit:** Code-level review of Notebooks 08 and 09 found no obvious direct outcome leakage or forward-looking tax-year feature calculations. The selected model uses 18 financial, governance, historical-trend, and derived-indicator predictors. Training used tax years 2010–2020, validation used 2021–2022, and later evaluation used 2023–2024. Unknown labels were excluded from supervised fitting and evaluation, and model preprocessing was fitted on training data.

**Outstanding limitations:** The evaluation is retrospective rather than a filing-publication-date simulation. Same-year predictors do not establish prospective prediction capability. Additional limitations include duplicate-filing and label-resolution issues, repeated organizations across temporal splits, negative downsampling, uncalibrated scores, and the lack of independent investigative review. The 2023–2024 period has already been examined during subsequent development and is not a pristine independent holdout for future model comparisons.

**Baseline artifacts and handoff:** The technical snapshot is preserved under `restricted_handoff/baseline_v1/`, including the model, scoring data, limited-data coverage, case profiles, separate evaluation labels, and supporting provenance information. Outcome labels must remain separate from prospective scoring inputs. Databricks ingestion and STTR experiments have not yet been implemented.

**Baseline decision:** Retain the current technically snapshotted baseline as the designated version 1 exploratory comparison point for future Databricks demonstrations and STTR experiments. Before relying on it as an immutable reference, preserve the complete artifact set and verify its checksums. Formal release approval remains separate from the analytical recommendation. This is an analytical recommendation, not an assertion that the technical release has received formal approval. The current snapshot status remains `SNAPSHOT_CREATED_EVALUATION_COMPUTED_PENDING_MANUAL_FEATURE_LINEAGE_APPROVAL`; any formal change to that status should be separately recorded and authorized.

**Next phase:** Load the preserved baseline scores, supporting data, and restricted case profiles into the Databricks demo environment. Validate row counts, schema, access restrictions, and version metadata after ingestion. Evaluate future STTR methods against the same baseline while clearly distinguishing observed-disclosure results from simulations and investigative-usefulness assessments.